# KLUE-RoBERTa — 낚시성 기사 탐지 학습

| 항목 | 값 |
|------|----|
| 모델 | `klue/roberta-base` |
| 토크나이저 | BPE (vocab 32,000) |
| token_type_ids | 미사용 (RoBERTa 계열) |
| 실행 환경 | Colab Pro V100/A100 |
| 예상 시간 | 이진 약 7.5시간 + 다중 약 2시간 |

In [ ]:
# 이 노트북이 학습할 모델 — 변경하지 않는다
MODEL_KEY = 'klue'

## Step 0. 패키지 설치

In [ ]:
!pip install -q transformers torch sentencepiece protobuf scikit-learn tqdm

## Step 1. Google Drive 마운트 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR = '/content/drive/MyDrive/text-mining-2026/data/processed'
SAVE_DIR = '/content/drive/MyDrive/text-mining-2026/models'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'데이터 경로: {DATA_DIR}')
print(f'저장 경로:   {SAVE_DIR}')

## Step 2. 라이브러리 임포트

In [ ]:
import json
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 3. 데이터 로딩 + 중복 제거

In [ ]:
# common_preprocess.py가 생성한 parquet 3개를 병합
# test_final.parquet은 봉인 — 절대 로딩하지 않음
df = pd.concat([
    pd.read_parquet(f'{DATA_DIR}/work_pool_clickbait_auto.parquet'),
    pd.read_parquet(f'{DATA_DIR}/work_pool_clickbait_direct.parquet'),
    pd.read_parquet(f'{DATA_DIR}/work_pool_nonclickbait_auto.parquet'),
], ignore_index=True)

# EDA §7: 제목+본문 동일한 중복 251건 → K-Fold data leakage 방지
before = len(df)
df = df.drop_duplicates(subset=['title_clean', 'content_clean']).reset_index(drop=True)
print(f'중복 제거: {before:,} → {len(df):,}건 ({before - len(df)}건 제거)')
print(f'컬럼: {list(df.columns)}')

## Step 4. 설정값

In [ ]:
MODEL_NAMES = {
    'kobert':    'skt/kobert-base-v1',
    'klue':      'klue/roberta-base',
    'koelectra': 'monologg/koelectra-base-v3-discriminator',
}

# KLUE-RoBERTa는 RoBERTa 계열 → token_type_ids 미사용
MODEL_USE_TTI = {
    'kobert':    False,
    'klue':      False,  # RoBERTa 계열 — segment embedding 없음
    'koelectra': True,
}

TYPE_NAMES = {
    0: '의문유발-부호', 1: '의문유발-은닉', 2: '선정표현',
    3: '속어/줄임말',  4: '사실과대',      5: '주어왜곡',
}

BATCH_SIZE    = 64      # A100 40GB / T4 16GB + bf16 기준
MAX_LENGTH    = 512
LR            = 2e-5
WARMUP_RATIO  = 0.1
MAX_GRAD_NORM = 1.0
WEIGHT_DECAY  = 0.01
EPOCHS        = {'binary': 3, 'multi': 5}
PATIENCE      = 2
N_FOLDS       = 5
RANDOM_STATE  = 42

print(f'학습 모델: {MODEL_KEY} ({MODEL_NAMES[MODEL_KEY]})')
print(f'BATCH={BATCH_SIZE}, LR={LR}, MAX_LEN={MAX_LENGTH}')

## Step 5. ClickbaitDataset

In [ ]:
class ClickbaitDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=MAX_LENGTH, task='binary'):
        if task == 'multi':
            dataframe = dataframe[dataframe['type_label'] != -1].reset_index(drop=True)
        self.titles   = dataframe['title_clean'].tolist()
        self.contents = dataframe['content_clean'].tolist()
        self.labels   = dataframe[
            'binary_label' if task == 'binary' else 'type_label'
        ].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.task       = task

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            text=self.titles[idx],
            text_pair=self.contents[idx],
            truncation='only_second',  # 제목 보존, 본문만 자름 (EDA §4)
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoded:
            item['token_type_ids'] = encoded['token_type_ids'].squeeze(0)
        else:
            item['token_type_ids'] = torch.zeros_like(item['input_ids'])
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print('ClickbaitDataset 정의 완료')

## Step 6. 헬퍼 함수

In [ ]:
def get_dataset(df, model_key, task='binary'):
    tok = AutoTokenizer.from_pretrained(MODEL_NAMES[model_key])
    return ClickbaitDataset(df, tok, task=task)

def get_fold_dataloaders(dataset, train_idx, val_idx):
    train_loader = DataLoader(
        Subset(dataset, train_idx), batch_size=BATCH_SIZE,
        shuffle=True, num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        Subset(dataset, val_idx), batch_size=BATCH_SIZE,
        shuffle=False, num_workers=2, pin_memory=True,
    )
    return train_loader, val_loader

def get_class_weights(labels, device='cpu'):
    classes = np.array(sorted(set(labels)))
    weights = compute_class_weight('balanced', classes=classes, y=np.array(labels))
    return torch.tensor(weights, dtype=torch.float).to(device)

def compute_metrics(labels, preds):
    return {
        'accuracy':        accuracy_score(labels, preds),
        'f1_macro':        f1_score(labels, preds, average='macro', zero_division=0),
        'precision_macro': precision_score(labels, preds, average='macro', zero_division=0),
        'recall_macro':    recall_score(labels, preds, average='macro', zero_division=0),
    }

print('헬퍼 함수 정의 완료')

## Step 7. 학습 / 평가 함수

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, model_key):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    for batch in tqdm(loader, desc='  train', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels         = batch['labels'].to(device)
        optimizer.zero_grad()
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if MODEL_USE_TTI[model_key]:
            kwargs['token_type_ids'] = token_type_ids
        with torch.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**kwargs)
            loss = criterion(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        all_preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), f1_score(all_labels, all_preds, average='macro', zero_division=0)

def eval_epoch(model, loader, criterion, model_key):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  val  ', leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels         = batch['labels'].to(device)
            kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
            if MODEL_USE_TTI[model_key]:
                kwargs['token_type_ids'] = token_type_ids
            with torch.autocast('cuda', dtype=torch.bfloat16):
                outputs = model(**kwargs)
                loss = criterion(outputs.logits, labels)
            total_loss += loss.item()
            all_preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    metrics = compute_metrics(all_labels, all_preds)
    metrics['loss'] = total_loss / len(loader)
    return metrics, np.array(all_preds), np.array(all_labels)

print('train_epoch / eval_epoch 정의 완료')

## Step 8. K-Fold 학습 함수

In [ ]:
def run_kfold(df, model_key, task='binary'):
    num_labels = 2 if task == 'binary' else 6
    n_epochs   = EPOCHS[task]

    result_path = f'{SAVE_DIR}/results_{model_key}_{task}.json'
    if os.path.exists(result_path):
        print(f'[{model_key}/{task}] 이미 완료 → 로드 후 스킵')
        with open(result_path, 'r', encoding='utf-8') as f:
            return json.load(f)['fold_results']

    print(f"\n{'='*60}")
    print(f'  {model_key} | {task} | num_labels={num_labels} | epochs={n_epochs}')
    print(f"{'='*60}")

    dataset    = get_dataset(df, model_key, task=task)
    labels_arr = np.array(dataset.labels)
    print(f'  Dataset 크기: {len(dataset):,}건')

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_results, fold_cms = [], []

    for fold, (train_idx, val_idx) in enumerate(skf.split(range(len(dataset)), labels_arr)):
        print(f'\n  ── Fold {fold+1}/{N_FOLDS} ──')
        print(f'     Train: {len(train_idx):,} | Val: {len(val_idx):,}')

        train_loader, val_loader = get_fold_dataloaders(dataset, train_idx, val_idx)

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAMES[model_key], num_labels=num_labels,
        ).to(device)

        if task == 'binary':
            criterion = nn.CrossEntropyLoss()
        else:
            weights = get_class_weights(labels_arr[train_idx].tolist(), device=device)
            criterion = nn.CrossEntropyLoss(weight=weights)
            print(f'     weights: {[f"{w:.3f}" for w in weights.cpu().numpy()]}')

        optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        total_steps  = len(train_loader) * n_epochs
        warmup_steps = int(total_steps * WARMUP_RATIO)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
        )
        print(f'     총 스텝: {total_steps:,} | warmup: {warmup_steps:,}')

        best_f1, best_metrics, best_preds, best_labels_fold = -1.0, None, None, None
        patience_cnt = 0
        save_path = f'{SAVE_DIR}/{model_key}_{task}_fold{fold+1}_best.pt'

        for epoch in range(n_epochs):
            t0 = time.time()
            train_loss, train_f1 = train_epoch(
                model, train_loader, optimizer, scheduler, criterion, model_key
            )
            val_metrics, val_preds, val_labels_fold = eval_epoch(
                model, val_loader, criterion, model_key
            )
            elapsed = time.time() - t0
            print(f'     Epoch {epoch+1}/{n_epochs} ({elapsed:.0f}s) | '
                  f'train loss={train_loss:.4f} f1={train_f1:.4f} | '
                  f'val loss={val_metrics["loss"]:.4f} f1={val_metrics["f1_macro"]:.4f} '
                  f'acc={val_metrics["accuracy"]:.4f}')

            if val_metrics['f1_macro'] > best_f1:
                best_f1, best_metrics = val_metrics['f1_macro'], val_metrics.copy()
                best_preds, best_labels_fold = val_preds.copy(), val_labels_fold.copy()
                patience_cnt = 0
                torch.save(model.state_dict(), save_path)
                print(f'     ✅ Best F1 갱신: {best_f1:.4f} → 저장')
            else:
                patience_cnt += 1
                if task == 'multi' and patience_cnt >= PATIENCE:
                    print(f'     ⏹ Early stopping')
                    break

        fold_results.append(best_metrics)
        fold_cms.append(confusion_matrix(best_labels_fold, best_preds))
        print(f'     Fold {fold+1} Best → '
              f'Acc={best_metrics["accuracy"]:.4f} | '
              f'F1={best_metrics["f1_macro"]:.4f} | '
              f'Prec={best_metrics["precision_macro"]:.4f} | '
              f'Rec={best_metrics["recall_macro"]:.4f}')

        del model
        torch.cuda.empty_cache()

    print(f"\n  {'='*50}")
    print(f'  [{model_key} / {task}] 5-Fold 요약')
    for metric in ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']:
        vals = [r[metric] for r in fold_results]
        print(f'  {metric:<22}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')

    if task == 'multi':
        total_cm = sum(fold_cms)
        print(f'\n  5-Fold 합산 Confusion Matrix:')
        header = ''.join([f'{TYPE_NAMES[i][:5]:>10}' for i in range(6)])
        print(f'  {"":>14}{header}')
        for i, row in enumerate(total_cm):
            print(f'  {TYPE_NAMES[i][:12]:<14}' + ''.join([f'{v:>10,}' for v in row]))
        n_11, confused = total_cm[0].sum(), total_cm[0][1]
        print(f'\n  [핵심 오분류] 부호→은닉: {confused:,}/{n_11:,} ({confused/n_11*100:.1f}%)')
        print(f'\n  Classification Report (마지막 fold):')
        print(classification_report(
            best_labels_fold, best_preds,
            target_names=list(TYPE_NAMES.values()), zero_division=0,
        ))

    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump({'model_key': model_key, 'task': task, 'fold_results': fold_results},
                  f, ensure_ascii=False, indent=2)
    print(f'  결과 저장: {result_path}')
    return fold_results

print('run_kfold 정의 완료')

## Step 9. 학습 전 검증 (Pre-flight Check)

아래 셀의 모든 항목이 ✅ 인지 확인한 후 학습을 시작합니다.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 이 셀의 모든 항목이 ✅ 여야 아래 학습 셀을 실행합니다.
# ❌ 또는 ⚠️ 항목은 해결 후 재실행하세요.
# ─────────────────────────────────────────────────────────────────────────────
import tempfile, warnings
warnings.filterwarnings('ignore')

errors = []
print(f"{'='*60}")
print(f'  Pre-flight Check — {MODEL_KEY}')
print(f"{'='*60}\n")

# ── 1. GPU ───────────────────────────────────────────────────────
print('[ 1 ] GPU')
if not torch.cuda.is_available():
    errors.append('GPU 없음 — Colab 런타임 유형 → GPU 로 변경 필요')
    print('  ❌ GPU 없음')
else:
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    if gpu_mem < 14.0:
        # BATCH_SIZE=16 기준 V100(16GB) 필요. T4(15GB) 간당간당, K80(12GB) OOM
        errors.append(f'GPU 메모리 {gpu_mem:.1f}GB — 14GB 미만. BATCH_SIZE=8 로 낮추거나 Colab Pro+ 런타임 재할당')
        print(f'  ⚠️  {gpu_name} {gpu_mem:.1f}GB — 14GB 미만, BATCH_SIZE 줄이기 검토')
    else:
        print(f'  ✅ {gpu_name} {gpu_mem:.1f}GB')

# ── 2. 데이터 파일 존재 ──────────────────────────────────────────
print('\n[ 2 ] DATA_DIR 파일')
required_files = [
    'work_pool_clickbait_auto.parquet',
    'work_pool_clickbait_direct.parquet',
    'work_pool_nonclickbait_auto.parquet',
]
for fname in required_files:
    fpath = f'{DATA_DIR}/{fname}'
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  ✅ {fname} ({size_mb:.1f}MB)')
    else:
        errors.append(f'파일 없음: {fpath}')
        print(f'  ❌ {fname} — Drive에 업로드됐는지 확인')

# ── 3. SAVE_DIR 쓰기 권한 ────────────────────────────────────────
print('\n[ 3 ] SAVE_DIR 쓰기 권한')
try:
    tmp = tempfile.NamedTemporaryFile(dir=SAVE_DIR, delete=True)
    tmp.close()
    print(f'  ✅ {SAVE_DIR}')
except Exception as e:
    errors.append(f'SAVE_DIR 쓰기 실패: {e}')
    print(f'  ❌ {e}')

# ── 4. 데이터 무결성 ─────────────────────────────────────────────
print('\n[ 4 ] 데이터 무결성')
n_total = len(df)
# 중복 제거 후 기대값: ~291,215건 (EDA §7: 251건 제거)
if n_total < 290_000 or n_total > 295_000:
    errors.append(f'레코드 수 이상: {n_total:,}건 (기대 ~291,215건)')
    print(f'  ❌ 레코드 수: {n_total:,}건 — 기대값 ~291,215')
else:
    print(f'  ✅ 레코드 수: {n_total:,}건')

for col in ['title_clean', 'content_clean', 'binary_label', 'type_label']:
    if col not in df.columns:
        errors.append(f'컬럼 없음: {col}')
        print(f'  ❌ 컬럼 없음: {col}')

for col in ['title_clean', 'content_clean']:
    null_cnt  = df[col].isna().sum()
    empty_cnt = (df[col] == '').sum()
    if null_cnt > 0 or empty_cnt > 0:
        errors.append(f'{col}: null={null_cnt}, 빈문자={empty_cnt} — common_preprocess 재실행 검토')
        print(f'  ❌ {col}: null={null_cnt}, 빈문자={empty_cnt}')
    else:
        print(f'  ✅ {col}: null=0, 빈문자=0')

# ── 5. 이진 레이블 분포 (EDA §1: 49.9:50.1) ─────────────────────
print('\n[ 5 ] 이진 레이블 분포 (기대: ~50:50)')
bin_counts = df['binary_label'].value_counts().sort_index()
bin_ratio  = bin_counts.get(1, 0) / n_total * 100
print(f'  정상(0): {bin_counts.get(0,0):,}건  낚시(1): {bin_counts.get(1,0):,}건  낚시 비율: {bin_ratio:.1f}%')
if abs(bin_ratio - 50.0) > 5.0:
    errors.append(f'이진 분포 이상: 낚시 비율 {bin_ratio:.1f}% (45~55% 기대)')
    print(f'  ❌ 균형 범위 초과 — class_weight 사용 검토')
else:
    print(f'  ✅ 균형 정상 → CrossEntropyLoss() (가중치 없음) 적용 예정')

# ── 6. 다중 분류 데이터 (EDA §2: Clickbait_Direct, 7.1x 불균형) ──
print('\n[ 6 ] 다중 분류 데이터 (type_label ≥ 0)')
df_multi = df[df['type_label'] != -1]
n_multi  = len(df_multi)
if abs(n_multi - 40_106) > 300:
    errors.append(f'다중 분류 레코드 수 이상: {n_multi:,}건 (기대 ~40,106건)')
    print(f'  ❌ 레코드 수: {n_multi:,}건 — 기대값 ~40,106')
else:
    print(f'  ✅ 레코드 수: {n_multi:,}건')

type_counts = df_multi['type_label'].value_counts().sort_index()
TYPE_NAMES_LOCAL = {
    0: '의문유발-부호', 1: '의문유발-은닉', 2: '선정표현',
    3: '속어/줄임말',  4: '사실과대',      5: '주어왜곡',
}
for lbl, cnt in type_counts.items():
    print(f'    {TYPE_NAMES_LOCAL.get(lbl, lbl)} ({lbl}): {cnt:,}건 ({cnt/n_multi*100:.1f}%)')
if len(type_counts) == 6:
    imbalance = type_counts.max() / type_counts.min()
    if imbalance > 25:
        errors.append(f'다중 분류 불균형 심각: {imbalance:.1f}x')
        print(f'  ❌ 불균형 {imbalance:.1f}x — 너무 심각')
    else:
        print(f'  ✅ 불균형 비율 {imbalance:.1f}x → CrossEntropyLoss(weight=...) 적용 예정')
else:
    errors.append(f'type_label 클래스 수 이상: {len(type_counts)}개 (기대 6개)')
    print(f'  ❌ 클래스 수: {len(type_counts)}개 (기대 6개)')

# ── 7. 토크나이저 로드 + truncation 동작 ─────────────────────────
print(f'\n[ 7 ] 토크나이저 ({MODEL_NAMES[MODEL_KEY]})')
try:
    tok = AutoTokenizer.from_pretrained(MODEL_NAMES[MODEL_KEY])
    # truncation='only_second': 제목(첫 번째) 보존, 본문(두 번째) 자름 (EDA §4)
    enc = tok('제목 테스트', '본문 테스트 ' * 100,
               truncation='only_second', max_length=MAX_LENGTH,
               padding='max_length', return_tensors='pt')
    assert enc['input_ids'].shape == (1, MAX_LENGTH), f'shape 오류: {enc["input_ids"].shape}'
    print(f'  ✅ vocab={tok.vocab_size:,}, truncation=only_second 동작 확인')
except Exception as e:
    errors.append(f'토크나이저 오류: {e}')
    print(f'  ❌ {e}')
    tok = None

# ── 8. ClickbaitDataset 인스턴스화 + shape ───────────────────────
print('\n[ 8 ] ClickbaitDataset')
ds_bin = ds_multi_test = None
if tok is not None:
    try:
        ds_bin    = ClickbaitDataset(df.head(200), tok, task='binary')
        s         = ds_bin[0]
        assert s['input_ids'].shape == (MAX_LENGTH,)
        assert s['labels'].item() in [0, 1]
        print(f'  ✅ binary: {len(ds_bin):,}건, shape={s["input_ids"].shape}')

        df_ms = df[df['type_label'] != -1].head(200).copy()
        ds_multi_test = ClickbaitDataset(df_ms, tok, task='multi')
        sm = ds_multi_test[0]
        assert sm['input_ids'].shape == (MAX_LENGTH,)
        assert 0 <= sm['labels'].item() <= 5
        print(f'  ✅ multi:  {len(ds_multi_test):,}건, shape={sm["input_ids"].shape}')
    except Exception as e:
        errors.append(f'Dataset 오류: {e}')
        print(f'  ❌ {e}')

# ── 9. DataLoader 배치 shape ─────────────────────────────────────
print('\n[ 9 ] DataLoader 배치 shape')
if ds_bin is not None:
    try:
        loader_test = DataLoader(ds_bin, batch_size=4, shuffle=False)
        batch = next(iter(loader_test))
        assert batch['input_ids'].shape    == (4, MAX_LENGTH)
        assert batch['attention_mask'].shape == (4, MAX_LENGTH)
        assert batch['labels'].shape        == (4,)
        print(f'  ✅ input_ids={list(batch["input_ids"].shape)}, '
              f'attention_mask={list(batch["attention_mask"].shape)}, '
              f'labels={list(batch["labels"].shape)}')
    except Exception as e:
        errors.append(f'DataLoader 오류: {e}')
        print(f'  ❌ {e}')

# ── 10. 클래스 가중치 계산 (EDA §2: 다중 분류 7.1x 불균형) ─────────
print('\n[ 10 ] 클래스 가중치')
try:
    sample_labels = df_multi['type_label'].tolist()
    cw = get_class_weights(sample_labels, device='cpu')
    assert len(cw) == 6
    ratio = cw.max().item() / cw.min().item()
    print(f'  ✅ weights={[f"{w:.3f}" for w in cw.numpy()]}')
    print(f'     max/min={ratio:.1f}x — 소수 클래스 보상 확인')
except Exception as e:
    errors.append(f'클래스 가중치 오류: {e}')
    print(f'  ❌ {e}')

# ── 11. token_type_ids 설정 확인 ─────────────────────────────────
print('\n[ 11 ] token_type_ids 설정')
if tok is not None:
    tti_flag = MODEL_USE_TTI[MODEL_KEY]
    has_tti  = 'token_type_ids' in tok('a', 'b', return_tensors='pt')
    if tti_flag and not has_tti:
        errors.append(f'{MODEL_KEY}: MODEL_USE_TTI=True지만 토크나이저가 token_type_ids 미반환')
        print(f'  ❌ 불일치 — MODEL_USE_TTI={tti_flag}, 토크나이저 반환={has_tti}')
    elif not tti_flag and has_tti:
        # RoBERTa: token_type_ids 반환하더라도 무시 (정상)
        print(f'  ✅ MODEL_USE_TTI[{MODEL_KEY}]={tti_flag} (RoBERTa 계열: token_type_ids 무시)')
    else:
        print(f'  ✅ MODEL_USE_TTI[{MODEL_KEY}]={tti_flag}')

# ── 최종 ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
if errors:
    print(f'  ❌ Pre-flight FAIL ({len(errors)}개 오류)\n')
    for i, e in enumerate(errors, 1):
        print(f'  {i}. {e}')
    print(f"{'='*60}")
    raise RuntimeError('Pre-flight check 실패 — 위 오류 해결 후 재실행')
else:
    print('  ✅ Pre-flight PASS — 모든 항목 정상')
    print(f'     모델  : {MODEL_KEY} ({MODEL_NAMES[MODEL_KEY]})')
    print(f'     이진  : {n_total:,}건, {bin_ratio:.1f}% 낚시')
    print(f'     다중  : {n_multi:,}건, 6클래스')
    print(f'     GPU   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음"}')
    print(f'  ➡️  Step 10 (이진 분류) 실행 가능')
print(f"{'='*60}")

## Step 9. 이진 분류 학습

- EDA §1: 49.9:50.1 균형 → class_weight 없음
- epochs=3, 291K건

In [ ]:
binary_results = run_kfold(df, model_key=MODEL_KEY, task='binary')

## Step 10. 다중 분류 학습

- EDA §2: 7.1배 불균형 → CrossEntropyLoss(weight=...) 적용
- EDA §6: 40K건, 과적합 주의 → early stopping (patience=2)
- epochs=5, Clickbait_Direct 40,106건만

In [ ]:
multi_results = run_kfold(df, model_key=MODEL_KEY, task='multi')

## Step 11. KLUE-RoBERTa 결과 요약

In [ ]:
for task, results in [('binary', binary_results), ('multi', multi_results)]:
    print(f'\n[KLUE-RoBERTa / {task}] 5-Fold 평균 ± 표준편차')
    for metric in ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']:
        vals = [r[metric] for r in results]
        print(f'  {metric:<22}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')